In [1]:
# Cell 1
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
interpretation_log = []

def evaluate_and_interpret(name, model, X_test, y_test, best_params=None, cv_score=None):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {'model': name, 'accuracy': acc, 'precision': prec,
               'recall': rec, 'f1': f1, 'auc': auc}
    results.append(metrics)

    recall_quality = "strong" if rec > 0.7 else "moderate" if rec > 0.5 else "weak"
    auc_quality = "strong" if auc > 0.8 else "moderate" if auc > 0.65 else "weak"

    md = f"""### {name}

**Best Hyperparameters:** {best_params if best_params else 'N/A'}
**CV Recall Score:** {f'{cv_score:.3f}' if cv_score else 'N/A'}

**Test Set Metrics:**
- Accuracy: {acc:.3f}
- Precision: {prec:.3f}
- Recall (Sensitivity): {rec:.3f}
- F1 Score: {f1:.3f}
- AUC: {auc:.3f}

**Confusion Matrix:** TN={tn}, FP={fp}, FN={fn}, TP={tp}

**Interpretation:**
- Correctly identifies {rec*100:.1f}% of truly anemic women (recall) — {recall_quality} for this health screening context.
- Of women predicted anemic, {prec*100:.1f}% actually are (precision).
- AUC of {auc:.3f} indicates {auc_quality} discriminative ability.
- False negatives (missed anemia cases): {fn}.
"""
    display(Markdown(md))
    interpretation_log.append(md)
    return metrics

In [2]:
with open('../models/ml/random_forest.pkl', 'rb') as f:
    best_rf = pickle.load(f)
with open('../models/ml/xgboost.pkl', 'rb') as f:
    best_xgb = pickle.load(f)
with open('../models/ml/lightgbm.pkl', 'rb') as f:
    best_lgbm = pickle.load(f)

print("Base models loaded: Random Forest, XGBoost, LightGBM")

xgb_params = best_xgb.get_params()
xgb_params.pop('early_stopping_rounds', None)
xgb_params['n_estimators'] = best_xgb.best_iteration

best_xgb_clean = XGBClassifier(**xgb_params)
best_xgb_clean.fit(X_train, y_train)

print(f"Clean XGBoost rebuilt with n_estimators={best_xgb.best_iteration}")

with open('../models/ml/xgboost.pkl', 'wb') as f:
    pickle.dump(best_xgb_clean, f)

Base models loaded: Random Forest, XGBoost, LightGBM
Clean XGBoost rebuilt with n_estimators=91


In [3]:
# Cell 3
base_learners = [
    ('rf', best_rf),
    ('xgb', best_xgb_clean),   # updated
    ('lgbm', best_lgbm)
]

meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=skf,
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)
evaluate_and_interpret('Stacking Ensemble (RF+XGB+LGBM → LogReg)', stacking_model, X_test, y_test)

### Stacking Ensemble (RF+XGB+LGBM → LogReg)

**Best Hyperparameters:** N/A
**CV Recall Score:** N/A

**Test Set Metrics:**
- Accuracy: 0.699
- Precision: 0.522
- Recall (Sensitivity): 0.549
- F1 Score: 0.535
- AUC: 0.706

**Confusion Matrix:** TN=355, FP=107, FN=96, TP=117

**Interpretation:**
- Correctly identifies 54.9% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 52.2% actually are (precision).
- AUC of 0.706 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 96.


{'model': 'Stacking Ensemble (RF+XGB+LGBM → LogReg)',
 'accuracy': 0.6992592592592592,
 'precision': 0.5223214285714286,
 'recall': 0.5492957746478874,
 'f1': 0.5354691075514875,
 'auc': 0.7060951567993822}

In [4]:
for name, model in [('Random Forest (standalone)', best_rf),
                     ('XGBoost (standalone)', best_xgb_clean),
                     ('LightGBM (standalone)', best_lgbm)]:
    evaluate_and_interpret(name, model, X_test, y_test)

### Random Forest (standalone)

**Best Hyperparameters:** N/A
**CV Recall Score:** N/A

**Test Set Metrics:**
- Accuracy: 0.710
- Precision: 0.541
- Recall (Sensitivity): 0.531
- F1 Score: 0.536
- AUC: 0.706

**Confusion Matrix:** TN=366, FP=96, FN=100, TP=113

**Interpretation:**
- Correctly identifies 53.1% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 54.1% actually are (precision).
- AUC of 0.706 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 100.


### XGBoost (standalone)

**Best Hyperparameters:** N/A
**CV Recall Score:** N/A

**Test Set Metrics:**
- Accuracy: 0.723
- Precision: 0.562
- Recall (Sensitivity): 0.554
- F1 Score: 0.558
- AUC: 0.699

**Confusion Matrix:** TN=370, FP=92, FN=95, TP=118

**Interpretation:**
- Correctly identifies 55.4% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 56.2% actually are (precision).
- AUC of 0.699 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 95.


### LightGBM (standalone)

**Best Hyperparameters:** N/A
**CV Recall Score:** N/A

**Test Set Metrics:**
- Accuracy: 0.689
- Precision: 0.506
- Recall (Sensitivity): 0.559
- F1 Score: 0.531
- AUC: 0.700

**Confusion Matrix:** TN=346, FP=116, FN=94, TP=119

**Interpretation:**
- Correctly identifies 55.9% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 50.6% actually are (precision).
- AUC of 0.700 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 94.


In [5]:
# Cell 5
output_path = '../data/processed/stacking_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved to {output_path}")

results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/stacking_metrics.csv', index=False)
print(results_df)

with open('../models/ml/stacking_ensemble.pkl', 'wb') as f:
    pickle.dump(stacking_model, f)

Saved to ../data/processed/stacking_interpretation.txt
                                      model  accuracy  precision    recall  \
0  Stacking Ensemble (RF+XGB+LGBM → LogReg)  0.699259   0.522321  0.549296   
1                Random Forest (standalone)  0.709630   0.540670  0.530516   
2                      XGBoost (standalone)  0.722963   0.561905  0.553991   
3                     LightGBM (standalone)  0.688889   0.506383  0.558685   

         f1       auc  
0  0.535469  0.706095  
1  0.535545  0.705882  
2  0.557920  0.698900  
3  0.531250  0.699988  
